# Linear Transformations - Lab

## Introduction

In this lab, you'll practice your linear transformation skills!

## Objectives

You will be able to:

* Determine if a linear transformation would be useful for a specific model or set of data
* Identify an appropriate linear transformation technique for a specific model or set of data
* Apply linear transformations to independent and dependent variables in linear regression
* Interpret the coefficients of variables that have been transformed using a linear transformation

## Ames Housing Data

Let's look at the Ames Housing data, where each record represents a home sale:

In [47]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

ames = pd.read_csv('ames.csv', index_col=0)
ames

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
Id,,,,,,,,,,,,,,,,,,,,,
1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500


We'll use this subset of features. These are specifically the _continuous numeric_ variables, which means that we'll hopefully have meaningful mean values.

From the data dictionary (`data_description.txt`):

```
LotArea: Lot size in square feet

MasVnrArea: Masonry veneer area in square feet

TotalBsmtSF: Total square feet of basement area

GrLivArea: Above grade (ground) living area square feet

GarageArea: Size of garage in square feet
```

In [48]:
ames = ames[[
    "LotArea",
    "MasVnrArea",
    "TotalBsmtSF",
    "GrLivArea",
    "GarageArea",
    "SalePrice"
]].copy()
ames

,LotArea,MasVnrArea,TotalBsmtSF,GrLivArea,GarageArea,SalePrice
Id,,,,,,
1,8450,196.0,856,1710,548,208500
2,9600,0.0,1262,1262,460,181500
3,11250,162.0,920,1786,608,223500
4,9550,0.0,756,1717,642,140000
5,14260,350.0,1145,2198,836,250000
...,...,...,...,...,...,...
1456,7917,0.0,953,1647,460,175000
1457,13175,119.0,1542,2073,500,210000
1458,9042,0.0,1152,2340,252,266500


We'll also drop any records with missing values for any of these features:

In [49]:
ames.dropna(inplace=True)
ames

,LotArea,MasVnrArea,TotalBsmtSF,GrLivArea,GarageArea,SalePrice
Id,,,,,,
1,8450,196.0,856,1710,548,208500
2,9600,0.0,1262,1262,460,181500
3,11250,162.0,920,1786,608,223500
4,9550,0.0,756,1717,642,140000
5,14260,350.0,1145,2198,836,250000
...,...,...,...,...,...,...
1456,7917,0.0,953,1647,460,175000
1457,13175,119.0,1542,2073,500,210000
1458,9042,0.0,1152,2340,252,266500


And plot the distributions of the un-transformed variables:

In [50]:
ames.hist(figsize=(15,10), bins="auto");

## Step 1: Build an Initial Linear Regression Model

`SalePrice` should be the target, and all other columns in `ames` should be predictors.

In [51]:
# Your code here - build a linear regression model with un-transformed features
import statsmodels.api as sm
#define the predictor and target
X =ames.drop(columns='SalePrice',axis=1)
y = ames['SalePrice']

#add the constant variable and fit the model
ames_initial_model=sm.OLS(y,sm.add_constant(X)).fit()

#Display the summary
ames_initial_model.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              SalePrice   R-squared:                       0.676
Model:                            OLS   Adj. R-squared:                  0.675
Method:                 Least Squares   F-statistic:                     603.0
Date:                Wed, 23 Apr 2025   Prob (F-statistic):               0.00
Time:                        16:43:41   Log-Likelihood:                -17622.
No. Observations:                1452   AIC:                         3.526e+04
Df Residuals:                    1446   BIC:                         3.529e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const       -1.525e+04   4145.934     -3.677      0.000   -2.34e+04   -7113.396
LotArea         0.2568      0.125      2.056      0.040       0.012       0.502
MasVnrArea     55.0481      7.427      7.412      0.000      40.480      69.616
TotalBsmtSF    44.1640      3.324     13.286      0.000      37.643      50.685
GrLivArea      63.8443      2.772     23.030      0.000      58.406      69.282
GarageArea     93.4629      6.795     13.755      0.000      80.134     106.792
==============================================================================
Omnibus:                      817.744   Durbin-Watson:                   1.991
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            77147.499
Skew:                          -1.709   Prob(JB):                         0.00
Kurtosis:                      38.546   Cond. No.                     5.09e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 5.09e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

## Step 2: Evaluate Initial Model and Interpret Coefficients

Describe the model performance overall and interpret the meaning of each predictor coefficient. Make sure to refer to the explanations of what each feature means from the data dictionary!

In [52]:
# Your written answer here
""" 
The model explains about 68% of the variance in Saleprice

The model is statistically significant overally with a F-statistic p-value of less than 0.05

The model coefficients are all statistically significant with t-statistic p-value well below 0.05

When all other variables are zero the expected sale of a house is $-1.525e+04 ,this however doesn't hold any meaning to the current scenario because that is not the legitimate price of the Sale price

LotArea: for every  increase in one square foot in  Lot size area, we expect an increase of $0.2568 in Sale Price of the house

MasVnrArea: for every increase in one square foot in masonry area size , we expect an increase of $55.0481 in Sale Price of the house

TotalBsmtSF:for every increase in one square foot of the the total square feet of the basement area we expect an increase of $44.1640 in Sale Price of the house

GrLivArea: for every increase of one square foot  of living area above ground, we expect an increase of $63.8443 in Sale Price of the house

GarageArea: for every increase in one square foot of  size of the size of garage , we expect an increse of $93.4629 in Sale price of the house





"""

" \nThe model explains about 68% of the variance in Saleprice\n\nThe model is statistically significant overally with a F-statistic p-value of less than 0.05\n\nThe model coefficients are all statistically significant with t-statistic p-value well below 0.05\n\nWhen all other variables are zero the expected sale of a house is $-1.525e+04 ,this however doesn't hold any meaning to the current scenario because that is not the legitimate price of the Sale price\n\nLotArea: for every  increase in one square foot in  Lot size area, we expect an increase of $0.2568 in Sale Price of the house\n\nMasVnrArea: for every increase in one square foot in masonry area size , we expect an increase of $55.0481 in Sale Price of the house\n\nTotalBsmtSF:for every increase in one square foot of the the total square feet of the basement area we expect an increase of $44.1640 in Sale Price of the house\n\nGrLivArea: for every increase of one square foot  of living area above ground, we expect an increase of 

<details>
    <summary style="cursor: pointer"><b>Answer (click to reveal)</b></summary>

The model overall is statistically significant and explains about 68% of the variance in sale price.

The coefficients are all statistically significant.

* `LotArea`: for each additional square foot of lot area, the price increases by about \\$0.26
* `MasVnrArea`: for each additional square foot of masonry veneer, the price increases by about \\$55
* `TotalBsmtSF`: for each additional square foot of basement area, the price increases by about \\$44
* `GrLivArea`: for each additional square foot of above-grade living area, the price increases by about \\$64
* `GarageArea`: for each additional square foot of garage area, the price increases by about \\$93

</details>

## Step 3: Express Model Coefficients in Metric Units

Your stakeholder gets back to you and says this is great, but they are interested in metric units.

Specifically they would like to measure area in square meters rather than square feet.

Report the same coefficients, except using square meters. You can do this by building a new model, or by transforming just the coefficients.

The conversion you can use is **1 square foot = 0.092903 square meters**.

In [53]:
# Your code here - building a new model or transforming coefficients
# from initial model so that they are in square meters
x_metric= X.copy()
x_metric = x_metric * 0.092903

new_ames_model= sm.OLS(y,sm.add_constant(x_metric)).fit()
new_ames_model.summary()




<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              SalePrice   R-squared:                       0.676
Model:                            OLS   Adj. R-squared:                  0.675
Method:                 Least Squares   F-statistic:                     603.0
Date:                Wed, 23 Apr 2025   Prob (F-statistic):               0.00
Time:                        16:43:41   Log-Likelihood:                -17622.
No. Observations:                1452   AIC:                         3.526e+04
Df Residuals:                    1446   BIC:                         3.529e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const       -1.525e+04   4145.934     -3.677      0.000   -2.34e+04   -7113.396
LotArea         2.7638      1.344      2.056      0.040       0.127       5.400
MasVnrArea    592.5326     79.941      7.412      0.000     435.719     749.346
TotalBsmtSF   475.3778     35.780     13.286      0.000     405.191     545.565
GrLivArea     687.2148     29.840     23.030      0.000     628.680     745.750
GarageArea   1006.0270     73.139     13.755      0.000     862.557    1149.497
==============================================================================
Omnibus:                      817.744   Durbin-Watson:                   1.991
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            77147.499
Skew:                          -1.709   Prob(JB):                         0.00
Kurtosis:                      38.546   Cond. No.                     4.73e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 4.73e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [54]:
print(f"""
      Initial model adjusted R-Squared:       {ames_initial_model.rsquared_adj}
      New model adjusted R_Squared:           {new_ames_model.rsquared_adj}
      
      """)


      Initial model adjusted R-Squared:       0.6747205931253312
      New model adjusted R_Squared:           0.6747205931253312
      
      


In [55]:
ames_initial_model.params


const         -15246.083611
LotArea            0.256769
MasVnrArea        55.048059
TotalBsmtSF       44.164028
GrLivArea         63.844321
GarageArea        93.462927
dtype: float64

In [56]:
new_ames_model.params

const         -15246.083611
LotArea            2.763844
MasVnrArea       592.532631
TotalBsmtSF      475.377849
GrLivArea        687.214850
GarageArea      1006.027003
dtype: float64

In [ ]:
# Your written answer here
""" 
The model explains about 68% of the variance in SalePrice meaning change in metrics did not affect  the adjusted variance
Change in metrics did not affect the significance of the model overally as it remained statistically significant with a F-statistic p-value well below 0.05%
Conversion of metrics from square feet to square meters did not affect the significance of coefficients,as it remained statistically significant with t-statistic p-value well below 0.05%  

When all other variables are zero the expected sale of a house is $-1.525e+04 ,this however doesn't hold any meaning to the current scenario because that is not the legitimate price of the Sale price

LotArea: for every  increase in one square meter in  Lot size area, we expect an increase of $2.763844 in Sale Price of the house

MasVnrArea: for every increase in one square meter in masonry area size , we expect an increase of $592.532631 in Sale Price of the house

TotalBsmtSF:for every increase in one square meter of the the total square feet of the basement area we expect an increase of $475.377849 in Sale Price of the house

GrLivArea: for every increase of one square meter of living area above ground, we expect an increase of $687.214850 in Sale Price of the house

GarageArea: for every increase in one square meter of  size of the size of garage , we expect an increse of $ 1006.027003 in Sale price of the house


"""


" \nThe model explains about 68% of the variance in SalePrice meaning change in metrics did not affect the the adjusted variance\nChange in metrics did not affect the significance of the model overally as it remained statistically significant with a F-statistic p-value well below 0.05%\nConversion of metrics from square feet to square meters did not affect the significance of coefficients,as it remained statistically significant with t-statistic p-value well below 0.05%  \n\nWhen all other variables are zero the expected sale of a house is $-1.525e+04 ,this however doesn't hold any meaning to the current scenario because that is not the legitimate price of the Sale price\n\nLotArea: for every  increase in one square meter in  Lot size area, we expect an increase of $2.763844 in Sale Price of the house\n\nMasVnrArea: for every increase in one square meter in masonry area size , we expect an increase of $592.532631 in Sale Price of the house\n\nTotalBsmtSF:for every increase in one squar

<details>
    <summary style="cursor: pointer"><b>Answer (click to reveal)</b></summary>

* `LotArea`: for each additional square meter of lot area, the price increases by about \\$2.76
* `MasVnrArea`: for each additional square meter of masonry veneer, the price increases by about \\$593
* `TotalBsmtArea`: for each additional square meter of basement area, the price increases by about \\$475
* `GrLivArea`: for each additional square meter of above-grade living area, the price increases by about \\$687
* `GarageArea`: for each additional square meter of garage area, the price increases by about \\$1,006

</details>

## Step 4: Center Data to Provide an Interpretable Intercept

Your stakeholder is happy with the metric results, but now they want to know what's happening with the intercept value. Negative \\$17k for a home with zeros across the board...what does that mean?

Center the data so that the mean is 0, fit a new model, and report on the new intercept.

(It doesn't matter whether you use data that was scaled to metric units or not. The intercept should be the same either way.)

In [58]:
X.describe()

,LotArea,MasVnrArea,TotalBsmtSF,GrLivArea,GarageArea
count,1452.000000,1452.000000,1452.000000,1452.000000,1452.000000
mean,10507.276171,103.685262,1055.847107,1514.091598,472.475207
std,9989.563592,181.066207,438.119089,525.627765,214.106397
min,1300.000000,0.000000,0.000000,334.000000,0.000000
25%,7538.750000,0.000000,794.750000,1128.000000,327.750000
50%,9478.500000,0.000000,990.500000,1461.500000,478.000000
75%,11600.000000,166.000000,1297.250000,1776.000000,576.000000
max,215245.000000,1600.000000,6110.000000,5642.000000,1418.000000


In [59]:
# Your code here - center data
X_centered = X.copy()

for col in X_centered.columns:
    X_centered[col] = X_centered[col]- X_centered[col].mean()

X_centered.describe()

,LotArea,MasVnrArea,TotalBsmtSF,GrLivArea,GarageArea
count,1.452000e+03,1.452000e+03,1.452000e+03,1.452000e+03,1.452000e+03
mean,5.612309e-13,5.637364e-15,-4.635166e-14,-6.890111e-14,1.879121e-14
std,9.989564e+03,1.810662e+02,4.381191e+02,5.256278e+02,2.141064e+02
min,-9.207276e+03,-1.036853e+02,-1.055847e+03,-1.180092e+03,-4.724752e+02
25%,-2.968526e+03,-1.036853e+02,-2.610971e+02,-3.860916e+02,-1.447252e+02
50%,-1.028776e+03,-1.036853e+02,-6.534711e+01,-5.259160e+01,5.524793e+00
75%,1.092724e+03,6.231474e+01,2.414029e+02,2.619084e+02,1.035248e+02
max,2.047377e+05,1.496315e+03,5.054153e+03,4.127908e+03,9.455248e+02


In [60]:
# Your code here - build a new model
X_centered_model= sm.OLS(y, sm.add_constant(X_centered)).fit()

X_centered_model.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              SalePrice   R-squared:                       0.676
Model:                            OLS   Adj. R-squared:                  0.675
Method:                 Least Squares   F-statistic:                     603.0
Date:                Wed, 23 Apr 2025   Prob (F-statistic):               0.00
Time:                        16:43:41   Log-Likelihood:                -17622.
No. Observations:                1452   AIC:                         3.526e+04
Df Residuals:                    1446   BIC:                         3.529e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const        1.806e+05   1186.695    152.200      0.000    1.78e+05    1.83e+05
LotArea         0.2568      0.125      2.056      0.040       0.012       0.502
MasVnrArea     55.0481      7.427      7.412      0.000      40.480      69.616
TotalBsmtSF    44.1640      3.324     13.286      0.000      37.643      50.685
GrLivArea      63.8443      2.772     23.030      0.000      58.406      69.282
GarageArea     93.4629      6.795     13.755      0.000      80.134     106.792
==============================================================================
Omnibus:                      817.744   Durbin-Watson:                   1.991
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            77147.499
Skew:                          -1.709   Prob(JB):                         0.00
Kurtosis:                      38.546   Cond. No.                     9.99e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 9.99e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [61]:
X_centered_model.params

const          180615.063361
LotArea             0.256769
MasVnrArea         55.048059
TotalBsmtSF        44.164028
GrLivArea          63.844321
GarageArea         93.462927
dtype: float64

In [62]:
# Your written answer here - interpret the new intercept
""" 
The model still explains about 68% of the variance in SalePrice
The model is still statistically significant overally with an F-stastistic p-value well below 0.05%
The coefficients of the model are still statistically significant with a t-statistic p-value well below 0.05%

When all the other variables are zero the expected sale of house is $180615.063361 which is positive from the other models,this means that a home with avaerage mean of Lot area,average masonry veneer area,average total basement area, average above-grade living area and average garage area would sell for about $180615.063361 

LotArea: for every  increase in one square foot in  Lot size area, we expect an increase of $0.256769 in Sale Price of the house

MasVnrArea: for every increase in one square foot in masonry area size , we expect an increase of $55.0481 in Sale Price of the house

TotalBsmtSF:for every increase in one square foot of the the total square feet of the basement area we expect an increase of $44.1640 in Sale Price of the house

GrLivArea: for every increase of one square foot  of living area above ground, we expect an increase of $63.8443 in Sale Price of the house

GarageArea: for every increase in one square foot of  size of the size of garage , we expect an increse of $93.4629 in Sale price of the house


"""


' \nThe model still explains about 68% of the variance in SalePrice\nThe model is still statistically significant overally with an F-stastistic p-value well below 0.05%\nThe coefficients of the model are still statistically significant with a t-statistic p-value well below 0.05%\n\nWhen all the other variables are zero the expected sale of house is $180615.063361 which is positive from the other models,this means that a home with avaerage mean of Lot area,average masonry veneer area,average total basement area, average above-grade living area and average garage area would sell for about $180615.063361 \n\nLotArea: for every  increase in one square foot in  Lot size area, we expect an increase of $0.256769 in Sale Price of the house\n\nMasVnrArea: for every increase in one square foot in masonry area size , we expect an increase of $55.0481 in Sale Price of the house\n\nTotalBsmtSF:for every increase in one square foot of the the total square feet of the basement area we expect an incre

<details>
    <summary style="cursor: pointer"><b>Answer (click to reveal)</b></summary>

The new intercept is about \\$181k. This means that a home with average lot area, average masonry veneer area, average total basement area, average above-grade living area, and average garage area would sell for about \\$181k.

</details>

## Step 5: Identify the "Most Important" Feature

Finally, either build a new model with transformed coefficients or transform the coefficients from the Step 4 model so that the most important feature can be identified.

Even though all of the features are measured in area, they are different kinds of area (e.g. lot area vs. masonry veneer area) that are not directly comparable as-is. So apply **standardization** (dividing predictors by their standard deviations) and identify the feature with the highest standardized coefficient as the "most important".

In [63]:
# Your code here - building a new model or transforming coefficients
# from centered model so that they are in standard deviations
X_standardized =X.copy()

for col in X_standardized:
    X_standardized[col] = (X_standardized[col]- X_standardized[col].mean()) \
          / X_standardized[col].std()
    
X_standardized.describe()   


,LotArea,MasVnrArea,TotalBsmtSF,GrLivArea,GarageArea
count,1.452000e+03,1.452000e+03,1.452000e+03,1.452000e+03,1.452000e+03
mean,5.138222e-17,9.787090e-18,-8.563704e-17,-1.529233e-16,9.297736e-17
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-9.216895e-01,-5.726373e-01,-2.409955e+00,-2.245109e+00,-2.206731e+00
25%,-2.971627e-01,-5.726373e-01,-5.959501e-01,-7.345343e-01,-6.759499e-01
50%,-1.029851e-01,-5.726373e-01,-1.491538e-01,-1.000548e-01,2.580396e-02
75%,1.093865e-01,3.441544e-01,5.509983e-01,4.982773e-01,4.835203e-01
max,2.049516e+01,8.263909e+00,1.153603e+01,7.853292e+00,4.416145e+00


In [64]:
standardized_model = sm.OLS(y, sm.add_constant(X_standardized)).fit()

standardized_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              SalePrice   R-squared:                       0.676
Model:                            OLS   Adj. R-squared:                  0.675
Method:                 Least Squares   F-statistic:                     603.0
Date:                Wed, 23 Apr 2025   Prob (F-statistic):               0.00
Time:                        16:43:41   Log-Likelihood:                -17622.
No. Observations:                1452   AIC:                         3.526e+04
Df Residuals:                    1446   BIC:                         3.529e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const        1.806e+05   1186.695    152.200      0.000    1.78e+05    1.83e+05
LotArea      2565.0144   1247.413      2.056      0.040     118.081    5011.948
MasVnrArea   9967.3432   1344.739      7.412      0.000    7329.495    1.26e+04
TotalBsmtSF  1.935e+04   1456.358     13.286      0.000    1.65e+04    2.22e+04
GrLivArea    3.356e+04   1457.171     23.030      0.000    3.07e+04    3.64e+04
GarageArea   2.001e+04   1454.817     13.755      0.000    1.72e+04    2.29e+04
==============================================================================
Omnibus:                      817.744   Durbin-Watson:                   1.991
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            77147.499
Skew:                          -1.709   Prob(JB):                         0.00
Kurtosis:                      38.546   Cond. No.                         2.19
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [65]:
standardized_model.params.sort_values(ascending = False)

const          180615.063361
GrLivArea       33558.347891
GarageArea      20011.010509
TotalBsmtSF     19349.103860
MasVnrArea       9967.343227
LotArea          2565.014370
dtype: float64

In [66]:
# Your written answer here - identify the "most important" feature
""" 
For each increase in 1 standard deviation of lot area , we see an increase in sale price of $2565.014370
For each increase in 1 standard deviation of MasVnrArea, we see an increase in sale price of $9967.343227
For each increase in 1 standard deviation of TotalBsmtSF, we see an increase in sale price of $19349.103860
For each increse in 1 standard deviation of GrLivArea, we see an increase in sale price of $33558.347891
For each increase in 1standard deviation of  GarageArea, we see an associated increase in sale price of $20011.010509

Comparing the variables coefficients, the most important feature is GrLivArea because it has the largest coefficient thus this means the above ground living is most important
"""

' \nFor each increase in 1 standard deviation of lot area , we see an increase in sale price of $2565.014370\nFor each increase in 1 standard deviation of MasVnrArea, we see an increase in sale price of $9967.343227\nFor each increase in 1 standard deviation of TotalBsmtSF, we see an increase in sale price of $19349.103860\nFor each increse in 1 standard deviation of GrLivArea, we see an increase in sale price of $33558.347891\nFor each increase in 1standard deviation of  GarageArea, we see an associated increase in sale price of $20011.010509\n\nComparing the variables coefficients, the most important feature is GrLivArea because it has the largest coefficient thus this means the above ground living is most important\n'

<details>
    <summary style="cursor: pointer"><b>Answer (click to reveal)</b></summary>

The feature with the highest standardized coefficient is `GrLivArea`. This means that above-grade living area is most important.

</details>

## Summary
Great! You've now got some hands-on practice transforming data and interpreting the results!